# The unsupervised approach

Supervised and unsupervised approaches answer fundamentally different questions. In the supervised case we are not trying to answer how likely a point in the data domain is, just how likely the oucome is. With the unsupervised approach we are able to model the tail more accurately and also infer correlation within the data set. 

-The unsupervised approach models the joint distribution $P(D_1, D_2, \dots, D_n)$.
-It generates entire loss distributions.
-It allows tail extrapolation and stress testing
-Does not require labels in the same sense

This is a credit risk model, while the supervised version is a classifier. In other words, the unsupervised learning can uncover hidden factor, sector clustering and concentration of risk.

# The model

Here we use an approach based on the publication https://arxiv.org/pdf/2202.11060, where the loss function corresponds to a Restricted Boltzmann Machine (RBM) associated with the following probability density: 
$$
p (v,h,\theta) = \frac{1}{Z} e^{-E(v,h;\theta)}\,,
$$
where 
$$
E(v,h;\theta) = -h^TWv - b^Tv - c^Th\,,
$$
is the so-called energy function, while $Z$ is called the partition function, which can be interpreted as a normalization constant. In fact, the model is inspired by statistical physics, where the partition function is
$$
Z = \sum_i e^{-\beta E_i}\,\qquad \beta = \frac{1}{k_B\,T}\,.
$$
Note that the weights are a matrix $W = W_{ij}\in \mathbb{R}^{m\times n}$, and the bias vectors are $b$ and $c$ (for each layer), with $\theta = (W,b,c)$. 

We import the data as in the supervised case from Kaggle competition Give Me Some Credit (https://www.kaggle.com/c/GiveMeSomeCredit):

In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# We load & prepare data (Give Me Some Credit) =====
# Put the CSV at: data/cs-training.csv
df = pd.read_csv("data/GiveMeSomeCredit/cs-training.csv", index_col = 0)

# Target and features

y_all = df["SeriousDlqin2yrs"].values.reshape(-1, 1)  # (N,1)
X_all = df.select_dtypes(include=[np.number]).values  # we keep numeric only

We now that the data has some undefined values for some entries (NaNs). In this case we have handle them differently as in the supervised learning. Here we do not replace those values by the median of the corresponding row, as this might induce false correlations 

In [11]:
X_df = df.drop(columns=["SeriousDlqin2yrs"], errors="ignore")

# number of rows with ≥1 NaN
n_rows_with_nan = X_df.isna().any(axis=1).sum() # Here we count the number of rows with at least one NaN
n_rows_total = len(X_df)

print(n_rows_with_nan, "rows with at least one NaN")
print(f"{n_rows_with_nan/n_rows_total:.2%} of rows")

29731 rows with at least one NaN
19.82% of rows


We re-scale the $X_\text{all}$ matrix and concatanate a mask matrix:

In [17]:
# Impute missing with median, then standardize

imputer = SimpleImputer(strategy="median") #Since the NaNs are undefined, we use the scikit function to replace the value with the median of that column (as a vector)
scaler  = StandardScaler()

Mdf = 1-X_df.isna().astype(int).values #we create the mas matrix


X_train, X_val, M_train, M_val, y_train, y_val = train_test_split( #We train
    X_all, Mdf, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

#stratify : we split the data in two sets (training and validation). Stratify ensures that both have the same shares of defaults

X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)

# Concatenate scaled X with UN-SCALED mask =====
X_train_final = np.concatenate([X_train_scaled, M_train], axis=1)
X_val_final   = np.concatenate([X_val_scaled,   M_val],   axis=1)

# shapes sanity check
print("X_train_final:", X_train_final.shape)
print("X_val_final:  ", X_val_final.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape)

X_train_final: (120000, 21)
X_val_final:   (30000, 21)
y_train: (120000, 1) y_val: (30000, 1)


In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: True
CUDA version: 12.1
GPU count: 1
GPU name: Quadro T2000


In [13]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import os
import math

# Convert numpy arrays to torch tensors
X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32 )
X_val_tensor   = torch.tensor(X_val_final, dtype=torch.float32 )

B=1024

# Create DataLoader for batching/shuffling
train_loader = DataLoader(X_train_tensor, batch_size=B, shuffle=True, drop_last=True)
val_loader   = DataLoader(X_val_tensor,   batch_size=B, shuffle=False)


In [16]:
"""
RBM (Gaussian visible, Bernoulli hidden) on Give Me Some Credit tabular data
- Builds missingness mask M from raw NaNs
- Splits BEFORE fitting imputer/scaler (prevents leakage)
- Fits imputer/scaler on train only, applies to val
- Concatenates scaled X with unscaled mask M
- Trains RBM with PCD-k
- Tracks reconstruction MSE on train/val

Run:
  python rbm_credit_pcd.py
"""

import torch
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------
# RBM Model (Gaussian-Bernoulli)
# -----------------------------
class GaussianBernoulliRBM(torch.nn.Module):
    def __init__(self, n_vis: int, n_hid: int, sigma: float = 1.0, device: str = "cpu"):
        super().__init__()
        self.n_vis = n_vis
        self.n_hid = n_hid
        self.sigma = float(sigma)
        self.device = torch.device(device)

        # Parameters: W (n_hid x n_vis), visible bias b_v (n_vis), hidden bias b_h (n_hid)
        # Small random init for W, zeros for biases
        W = 0.01 * torch.randn(n_hid, n_vis, device=self.device)
        b_v = torch.zeros(n_vis, device=self.device)
        b_h = torch.zeros(n_hid, device=self.device)

        self.W = torch.nn.Parameter(W)
        self.b_v = torch.nn.Parameter(b_v)
        self.b_h = torch.nn.Parameter(b_h)

    @torch.no_grad()
    def hidden_probs(self, v: torch.Tensor) -> torch.Tensor:
        # v: (B, n_vis)
        # p(h=1|v) = sigmoid(v W^T / sigma^2 + b_h)
        return torch.sigmoid((v @ self.W.t()) / (self.sigma ** 2) + self.b_h)

    @torch.no_grad()
    def sample_h_given_v(self, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        p_h = self.hidden_probs(v)
        h = torch.bernoulli(p_h)
        return p_h, h

    @torch.no_grad()
    def visible_mean(self, h: torch.Tensor) -> torch.Tensor:
        # mean(v|h) = h W + b_v
        return h @ self.W + self.b_v

    @torch.no_grad()
    def sample_v_given_h(self, h: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        mean_v = self.visible_mean(h)
        v = mean_v + self.sigma * torch.randn_like(mean_v)
        return mean_v, v

    @torch.no_grad()
    def gibbs_step(self, v: torch.Tensor) -> torch.Tensor:
        _, h = self.sample_h_given_v(v)
        _, v_new = self.sample_v_given_h(h)
        return v_new

    @torch.no_grad()
    def gibbs_k(self, v0: torch.Tensor, k: int) -> torch.Tensor:
        v = v0
        for _ in range(int(k)):
            v = self.gibbs_step(v)
        return v

    @torch.no_grad()
    def reconstruction_mse(self, v: torch.Tensor) -> torch.Tensor:
        # One-step reconstruction using mean (no noise) for a stable diagnostic
        p_h, _ = self.sample_h_given_v(v)
        h = p_h  # use probabilities rather than samples
        v_recon_mean = h @ self.W + self.b_v
        return torch.mean((v - v_recon_mean) ** 2)


# -----------------------------
# Training loop (PCD-k)
# -----------------------------
@torch.no_grad()
def train_pcd(
    rbm: GaussianBernoulliRBM,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 20,
    k: int = 1,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    persistent_init: str = "batch",  # "batch" or "noise"
):
    device = rbm.device
    rbm.train()

    # Persistent chain (tied to training batch size)
    first_batch = next(iter(train_loader))[0].to(device)
    B = first_batch.shape[0]

    if persistent_init == "noise":
        v_persist = torch.randn(B, rbm.n_vis, device=device)
    else:
        v_persist = first_batch.clone()

    for epoch in range(1, epochs + 1):
        rbm.train()
        train_mse_accum = 0.0
        train_batches = 0

        for (v_data,) in train_loader:
            v_data = v_data.to(device)
            if v_data.shape[0] != B:
                # If drop_last=True in train_loader, this shouldn't happen.
                # Skip to keep persistent chain shape consistent.
                continue

            # -------- Positive phase --------
            p_h_data = torch.sigmoid((v_data @ rbm.W.t()) / (rbm.sigma ** 2) + rbm.b_h)  # (B, n_hid)

            pos_W = (p_h_data.t() @ v_data) / B                 # (n_hid, n_vis)
            pos_bv = torch.mean(v_data, dim=0)                  # (n_vis,)
            pos_bh = torch.mean(p_h_data, dim=0)                # (n_hid,)

            # -------- Negative phase (PCD-k) --------
            v_model = rbm.gibbs_k(v_persist, k=k)
            v_persist = v_model.detach()

            p_h_model = torch.sigmoid((v_model @ rbm.W.t()) / (rbm.sigma ** 2) + rbm.b_h)

            neg_W = (p_h_model.t() @ v_model) / B
            neg_bv = torch.mean(v_model, dim=0)
            neg_bh = torch.mean(p_h_model, dim=0)

            # -------- Manual parameter update --------
            # Gradient ascent on log-likelihood approx: (pos - neg)
            rbm.W += lr * (pos_W - neg_W) - weight_decay * rbm.W
            rbm.b_v += lr * (pos_bv - neg_bv)
            rbm.b_h += lr * (pos_bh - neg_bh)

            # Diagnostics
            train_mse_accum += rbm.reconstruction_mse(v_data).item()
            train_batches += 1

        # Validation diagnostics
        rbm.eval()
        val_mse_accum = 0.0
        val_batches = 0
        for (v_val,) in val_loader:
            v_val = v_val.to(device)
            val_mse_accum += rbm.reconstruction_mse(v_val).item()
            val_batches += 1

        train_mse = train_mse_accum / max(1, train_batches)
        val_mse = val_mse_accum / max(1, val_batches)

        print(f"Epoch {epoch:03d} | train recon MSE: {train_mse:.6f} | val recon MSE: {val_mse:.6f}")


# -----------------------------
# Main script
# -----------------------------
def main():
    # ----- Config -----
    csv_path = "data/GiveMeSomeCredit/cs-training.csv"
    test_size = 0.2
    random_state = 42

    batch_size_train = 256
    batch_size_val = 512

    n_hid = 128
    epochs = 30
    k = 1                 # PCD-k
    lr = 5e-4
    weight_decay = 1e-4

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ----- Load -----
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path, index_col=0)

    # Target (only for stratified split / later evaluation)
    y_all = df["SeriousDlqin2yrs"].values

    # Raw features with NaNs preserved (numeric only)
    X_df = df.drop(columns=["SeriousDlqin2yrs"]).select_dtypes(include=[np.number])

    # Mask from original NaNs (1 = observed, 0 = missing)
    M_df = (~X_df.isna()).astype(np.int8)

    X_raw = X_df.values  # contains NaNs
    M = M_df.values      # 0/1

    # ----- Split FIRST (prevents leakage) -----
    X_train_raw, X_val_raw, M_train, M_val, y_train, y_val = train_test_split(
        X_raw, M, y_all,
        test_size=test_size,
        random_state=random_state,
        stratify=y_all
    )

    # ----- Fit preprocessing on TRAIN only -----
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(X_train_raw)
    X_val_imp = imputer.transform(X_val_raw)

    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)

    # ----- Concatenate scaled X with unscaled mask -----
    X_train_final = np.concatenate([X_train_scaled, M_train], axis=1).astype(np.float32)
    X_val_final = np.concatenate([X_val_scaled, M_val], axis=1).astype(np.float32)

    n_vis = X_train_final.shape[1]
    print(f"n_vis = {n_vis}, n_hid = {n_hid}, device = {device}")
    print(f"Train: {X_train_final.shape} | Val: {X_val_final.shape}")

    # ----- Torch datasets/loaders -----
    train_ds = TensorDataset(torch.from_numpy(X_train_final))
    val_ds = TensorDataset(torch.from_numpy(X_val_final))

    train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size_val, shuffle=False, drop_last=False)

    # ----- Model -----
    rbm = GaussianBernoulliRBM(n_vis=n_vis, n_hid=n_hid, sigma=1.0, device=device)

    # ----- Train -----
    train_pcd(
        rbm,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        k=k,
        lr=lr,
        weight_decay=weight_decay,
        persistent_init="batch",
    )

    # Optional: save model parameters
    torch.save(
        {"W": rbm.W.detach().cpu(), "b_v": rbm.b_v.detach().cpu(), "b_h": rbm.b_h.detach().cpu()},
        "rbm_params.pt",
    )
    print("Saved: rbm_params.pt")


if __name__ == "__main__":
    main()
